# DevGen — Devanagari Handwriting GAN

Generates synthetic handwritten Devanagari word images using a **ResNet GAN** with:
- **Hinge Loss** for stable adversarial training
- **DiffAugment** to prevent discriminator overfitting
- **Spectral Normalization** on discriminator weights
- **Exponential Moving Average (EMA)** for higher-quality inference
- **TTUR** (Two Time-scale Update Rule): D learns 4× faster than G

Inspired by Chhatkuli et al. (2021) and enhanced with techniques from BigGAN and StyleGAN2.

### Setup
1. **GPU**: Settings → Accelerator → **GPU T4 x2**
2. **Internet**: **ON** (downloads dataset from HuggingFace)
3. **Run All** — takes ~6–8 hours for 100k steps

In [ ]:
!pip install -q datasets

import copy, os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.utils import save_image
from datasets import load_dataset
from tqdm.notebook import tqdm
from PIL import Image

print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────
LATENT_DIM = 128
IMG_SIZE   = 64
CHANNELS   = 1
G_CH       = 64       # Generator channel multiplier
D_CH       = 64       # Discriminator channel multiplier
BATCH_SIZE = 64
STEPS      = 100000
N_CRITIC   = 2        # D updates per G update
SAMPLE_INT = 2000
EMA_DECAY  = 0.999

In [ ]:
# ── DiffAugment ─────────────────────────────────────────────────────────
def DiffAugment(x, policy='color,translation,cutout'):
    if not policy: return x
    for p in policy.split(','):
        for fn in _AUG[p]: x = fn(x)
    return x.contiguous()

def _rb(x): return x + (torch.rand(x.size(0),1,1,1, dtype=x.dtype, device=x.device) - 0.5)
def _rs(x):
    m = x.mean(1, keepdim=True)
    return (x-m) * (torch.rand(x.size(0),1,1,1, dtype=x.dtype, device=x.device)*2) + m
def _rc(x):
    m = x.mean([1,2,3], keepdim=True)
    return (x-m) * (torch.rand(x.size(0),1,1,1, dtype=x.dtype, device=x.device)*0.5+0.5) + m
def _rt(x, r=0.125):
    sx, sy = int(x.size(2)*r+.5), int(x.size(3)*r+.5)
    tx = torch.randint(-sx, sx+1, [x.size(0),1,1], device=x.device)
    ty = torch.randint(-sy, sy+1, [x.size(0),1,1], device=x.device)
    gb,gx,gy = torch.meshgrid(torch.arange(x.size(0),device=x.device), torch.arange(x.size(2),device=x.device), torch.arange(x.size(3),device=x.device), indexing='ij')
    gx, gy = torch.clamp(gx+tx+1,0,x.size(2)+1), torch.clamp(gy+ty+1,0,x.size(3)+1)
    return F.pad(x,[1,1,1,1]).permute(0,2,3,1).contiguous()[gb,gx,gy].permute(0,3,1,2)
def _rco(x, r=0.5):
    cs = int(x.size(2)*r+.5), int(x.size(3)*r+.5)
    ox = torch.randint(0, x.size(2)+(1-cs[0]%2), [x.size(0),1,1], device=x.device)
    oy = torch.randint(0, x.size(3)+(1-cs[1]%2), [x.size(0),1,1], device=x.device)
    gb,gx,gy = torch.meshgrid(torch.arange(x.size(0),device=x.device), torch.arange(cs[0],device=x.device), torch.arange(cs[1],device=x.device), indexing='ij')
    gx, gy = torch.clamp(gx+ox-cs[0]//2,0,x.size(2)-1), torch.clamp(gy+oy-cs[1]//2,0,x.size(3)-1)
    m = torch.ones(x.size(0),x.size(2),x.size(3), dtype=x.dtype, device=x.device)
    m[gb,gx,gy] = 0
    return x * m.unsqueeze(1)

_AUG = {'color': [_rb,_rs,_rc], 'translation': [_rt], 'cutout': [_rco]}

In [ ]:
# ── Architecture ────────────────────────────────────────────────────────

class ResBlockUp(nn.Module):
    """Generator: upsample + two convs with BN."""
    def __init__(self, ic, oc):
        super().__init__()
        self.bn1, self.c1 = nn.BatchNorm2d(ic), nn.Conv2d(ic, oc, 3, 1, 1)
        self.bn2, self.c2 = nn.BatchNorm2d(oc), nn.Conv2d(oc, oc, 3, 1, 1)
        self.sc = nn.Conv2d(ic, oc, 1) if ic != oc else nn.Identity()
    def forward(self, x):
        h = F.interpolate(F.relu(self.bn1(x)), scale_factor=2, mode='bilinear', align_corners=False)
        h = self.c2(F.relu(self.bn2(self.c1(h))))
        return h + self.sc(F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False))

class ResBlockDown(nn.Module):
    """Discriminator: two SN convs + avg pool."""
    def __init__(self, ic, oc, first=False):
        super().__init__()
        self.first = first
        self.c1 = nn.utils.spectral_norm(nn.Conv2d(ic, oc, 3, 1, 1))
        self.c2 = nn.utils.spectral_norm(nn.Conv2d(oc, oc, 3, 1, 1))
        self.sc = nn.utils.spectral_norm(nn.Conv2d(ic, oc, 1)) if ic != oc else nn.Identity()
    def forward(self, x):
        h = x if self.first else F.relu(x)
        return F.avg_pool2d(self.c2(F.relu(self.c1(h))) + self.sc(x), 2)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(LATENT_DIM, G_CH*8*4*4)
        self.blocks = nn.ModuleList([
            ResBlockUp(G_CH*8, G_CH*4),  # 4→8
            ResBlockUp(G_CH*4, G_CH*2),  # 8→16
            ResBlockUp(G_CH*2, G_CH),    # 16→32
            ResBlockUp(G_CH, G_CH),      # 32→64
        ])
        self.out = nn.Sequential(nn.BatchNorm2d(G_CH), nn.ReLU(True), nn.Conv2d(G_CH, CHANNELS, 3,1,1), nn.Tanh())
    def forward(self, z):
        h = self.fc(z).view(-1, G_CH*8, 4, 4)
        for b in self.blocks: h = b(h)
        return self.out(h)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList([
            ResBlockDown(CHANNELS, D_CH, first=True),
            ResBlockDown(D_CH, D_CH*2),
            ResBlockDown(D_CH*2, D_CH*4),
            ResBlockDown(D_CH*4, D_CH*8),
        ])
        self.out = nn.Sequential(nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.utils.spectral_norm(nn.Linear(D_CH*8, 1)))
    def forward(self, x):
        for b in self.blocks: x = b(x)
        return self.out(x)

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────
class HFWordDataset(Dataset):
    def __init__(self, hf_ds, transform):
        self.ds, self.tf = hf_ds, transform
    def __len__(self): return len(self.ds)
    def __getitem__(self, i): return self.tf(self.ds[i]['image'].convert('L')), 0

print("Downloading IIIT-INDIC-HW-WORDS-Hindi...")
hf_ds = load_dataset("c3rl/IIIT-INDIC-HW-WORDS-Hindi", split="train")
print(f"Loaded {len(hf_ds)} word images.")

tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])
loader = DataLoader(HFWordDataset(hf_ds, tf), batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)

In [ ]:
# ── Training ─────────────────────────────────────────────────────────────

class EMA:
    def __init__(self, model, decay=0.999):
        self.model = copy.deepcopy(model).eval()
        self.decay = decay
    @torch.no_grad()
    def update(self, model):
        for ep, mp in zip(self.model.parameters(), model.parameters()):
            ep.data.mul_(self.decay).add_(mp.data, alpha=1-self.decay)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

G = Generator().to(device)
D = Discriminator().to(device)
ema = EMA(G, EMA_DECAY)

opt_G = optim.Adam(G.parameters(), lr=1e-4, betas=(0.0, 0.9))
opt_D = optim.Adam(D.parameters(), lr=4e-4, betas=(0.0, 0.9))

fixed_z = torch.randn(16, LATENT_DIM, device=device)
os.makedirs('samples', exist_ok=True)

step = 0
pbar = tqdm(total=STEPS, desc='GAN Training')

while step < STEPS:
    for imgs, _ in loader:
        if step >= STEPS: break
        real = imgs.to(device)
        bs = real.size(0)

        # D step (×N_CRITIC)
        for _ in range(N_CRITIC):
            with torch.no_grad(): fake = G(torch.randn(bs, LATENT_DIM, device=device))
            ld = F.relu(1-D(DiffAugment(real))).mean() + F.relu(1+D(DiffAugment(fake))).mean()
            opt_D.zero_grad(); ld.backward(); opt_D.step()

        # G step
        fake = G(torch.randn(bs, LATENT_DIM, device=device))
        lg = -D(DiffAugment(fake)).mean()
        opt_G.zero_grad(); lg.backward(); opt_G.step()
        ema.update(G)

        step += 1; pbar.update(1)
        if step % 100 == 0: pbar.set_postfix(D=f"{ld.item():.2f}", G=f"{lg.item():.2f}")
        if step % SAMPLE_INT == 0:
            with torch.no_grad(): s = ema.model(fixed_z)
            save_image(s, f'samples/step_{step}.png', nrow=4, normalize=True)
            torch.save(ema.model.state_dict(), 'generator_ema.pth')

pbar.close()
print('Training complete!')

In [ ]:
# ── Generate Final Dataset ───────────────────────────────────────────────
NUM_GENERATE = 1000
os.makedirs('generated_dataset', exist_ok=True)

ema.model.eval()
with torch.no_grad():
    for i in tqdm(range(NUM_GENERATE), desc='Generating'):
        img = ema.model(torch.randn(1, LATENT_DIM, device=device))
        save_image(img, f'generated_dataset/gen_{i}.png', normalize=True)

!zip -r generated_dataset.zip generated_dataset
print(f'Done! {NUM_GENERATE} images saved and zipped.')